<a href="https://colab.research.google.com/github/AbhishekShah272002/Deep-Learning-and-NLP/blob/main/household_power_consumption_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [ ]:
data = pd.read_csv(
    "household_power_consumption.txt",
    sep=";",
    na_values=["?"],
    low_memory=False
)

In [ ]:
data["Datetime"] = pd.to_datetime(
    data["Date"] + " " + data["Time"],
    dayfirst=True,
    errors="coerce"
)

data = data.dropna(subset=["Datetime"])
data = data.sort_values("Datetime")
data = data.drop_duplicates(subset=["Datetime"])
data = data.set_index("Datetime")

In [ ]:
target_column = "Global_active_power"
data[target_column] = pd.to_numeric(
    data[target_column],
    errors="coerce"
)

In [ ]:
import joblib
import tensorflow as tf

# Load the saved model and scaler
model = tf.keras.models.load_model("gru_model.keras")
scaler = joblib.load("scaler.pkl")

In [ ]:
import os
os.listdir('.')

['.config', 'household_power_consumption.txt', 'sample_data']

In [ ]:
sampling_interval = "1min"


In [ ]:
data = data[[target_column]].resample(sampling_interval).mean()
data[target_column] = data[target_column].interpolate(
    method="time"
).ffill().bfill()

In [ ]:
import joblib
import tensorflow as tf

# Ensure model and scaler are loaded
model = tf.keras.models.load_model("gru_model.keras")
scaler = joblib.load("scaler.pkl")

scaled_data = scaler.transform(data[[target_column]])

In [ ]:
import numpy as np
import tensorflow as tf

# Ensure model is defined
try:
    model
except NameError:
    model = tf.keras.models.load_model("gru_model.keras")

window_length = model.input_shape[1]

X = []
y_true = []

for i in range(window_length, len(scaled_data)):
    X.append(scaled_data[i - window_length:i])
    y_true.append(scaled_data[i, 0])

X = np.asarray(X)
y_true = np.asarray(y_true)

In [ ]:
import tensorflow as tf

# Ensure model is defined
try:
    model
except NameError:
    model = tf.keras.models.load_model("gru_model.keras")

y_pred = model.predict(X, verbose=0)

In [ ]:
import joblib

# Ensure scaler is defined
try:
    scaler
except NameError:
    scaler = joblib.load("scaler.pkl")

y_pred_original = scaler.inverse_transform(
    y_pred.reshape(-1, 1)
).ravel()

y_true_original = scaler.inverse_transform(
    y_true.reshape(-1, 1)
).ravel()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_true_original, y_pred_original)
rmse = np.sqrt(mean_squared_error(y_true_original, y_pred_original))

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Actual_Global_active_power": y_true_original,
    "Predicted_Global_active_power": y_pred_original
}, index=data.index[window_length:])

metrics = pd.DataFrame({
    "Metric": ["MAE", "RMSE"],
    "Value": [mae, rmse],
    "Unit": ["kilowatts", "kilowatts"]
})

display(results.head(10))
display(metrics)

,Actual_Global_active_power,Predicted_Global_active_power
Datetime,,
2006-12-16 18:24:00,3.452,0.714301
2006-12-16 18:25:00,4.870,0.788197
2006-12-16 18:26:00,4.868,1.003245
2006-12-16 18:27:00,4.866,1.079903
2006-12-16 18:28:00,3.176,1.099450
2006-12-16 18:29:00,2.920,0.873949
2006-12-16 18:30:00,2.930,0.753771
2006-12-16 18:31:00,2.912,0.715249
2006-12-16 18:32:00,2.608,0.704880


,Metric,Value,Unit
0,MAE,0.698560,kilowatts
1,RMSE,1.108578,kilowatts
